In [0]:

MERGE INTO bootcamp_matias.silver.propiedades AS t
USING (
WITH 
--Limpieza básica y conversión de tipos
datos_limpios AS (
    SELECT 
        LOWER(TRIM(zona)) AS zona,
        
        -- Tipo de operación (normalizado)
        LOWER(TRIM(tipo_de_operacion)) AS tipo_operacion,
        
        -- Precio (validado)
        TRY_CAST(precio AS DECIMAL(10,2)) AS precio,
        UPPER(TRIM(moneda)) as moneda,
        COALESCE(TRY_CAST(expensas AS DECIMAL(10,2)), 0) AS expensas,
        
        -- Dimensiones
        CASE 
            WHEN CAST(TRY_CAST(ambientes AS FLOAT) AS INT) BETWEEN 1 AND 6 
            THEN CAST(TRY_CAST(ambientes AS FLOAT) AS INT)
            ELSE 0
        END AS ambientes,
        
        TRY_CAST(metros_cuadrados_totales AS DECIMAL(10,2)) as metros_cuadrados_totales,
        TRY_CAST(metros_cuadrados_cubiertos AS DECIMAL(10,2)) as metros_cuadrados_cubiertos,
      
        --Estado normalizado
        LOWER(TRIM(estado)) AS estado,
             -- Antigüedad (convertir 999)
        CASE 
            WHEN TRY_CAST(antiguedad AS INT) IS NULL THEN 999
            ELSE TRY_CAST(antiguedad AS INT) 
        END AS antiguedad,

        url
        
    FROM bootcamp_matias.raw.propiedades_bronze
    
    -- FILTROS DE CALIDAD: Excluir registros inválidos
    WHERE 
        -- Precio válido
        precio IS NOT NULL 
        AND TRY_CAST(precio AS DECIMAL(10,2)) > 0
        -- Metros cuadrados válidos
        AND metros_cuadrados_totales IS NOT NULL
        AND TRY_CAST(metros_cuadrados_totales AS DECIMAL(10,2)) > 0
        AND metros_cuadrados_cubiertos IS NOT NULL
        AND TRY_CAST(metros_cuadrados_cubiertos AS DECIMAL(10,2)) > 0
        -- Zona no vacía
        AND zona IS NOT NULL 
        AND TRIM(zona) != ''
        -- Tipo de operación válido
        AND tipo_de_operacion != 'null'
                
),

estandarizacion_columnas AS (

    SELECT

    REGEXP_EXTRACT(zona,'([a-z]+(?:-[a-z]+)*)$',1) AS ciudad,

    CASE
        WHEN zona = 'capital-federal' THEN 'caba'

        
        WHEN zona RLIKE 'bsas-gba-norte|gba-zona-norte' THEN 'gba_norte'
        WHEN zona RLIKE 'bsas-gba-sur|gba-zona-sur' THEN 'gba_sur'
        WHEN zona RLIKE 'bsas-gba-oeste|gba-zona-oeste' THEN 'gba_oeste'


        WHEN REGEXP_EXTRACT(zona,'([a-z]+(?:-[a-z]+)*)$',1)
            RLIKE 'tigre|pilar|vicente-lopez|san-isidro|escobar|san-fernando|san-miguel|jose-c-paz|malvinas-argentinas|general-san-martin'
            THEN 'gba_norte'

        WHEN REGEXP_EXTRACT(zona,'([a-z]+(?:-[a-z]+)*)$',1)
            RLIKE 'quilmes|lanus|avellaneda|lomas-de-zamora|berazategui|florencio-varela|almirante-brown|esteban-echeverria|ezeiza|presidente-peron|san-vicente|la-plata|ensenada|berisso'
            THEN 'gba_sur'

        WHEN REGEXP_EXTRACT(zona,'([a-z]+(?:-[a-z]+)*)$',1)
            RLIKE 'moron|la-matanza|ituzaingo|hurlingham|merlo|moreno|tres-de-febrero|general-rodriguez|lujan|marcos-paz|canuelas|castelar|caseros'
            THEN 'gba_oeste'

        ELSE 'otros'
    END AS zona,

    CASE 
        WHEN tipo_operacion RLIKE 'venta.*alquiler' THEN 'venta_alquiler'
        WHEN tipo_operacion RLIKE '^venta$'THEN 'venta'
        WHEN tipo_operacion RLIKE 'alquiler|alquier' THEN 'alquiler'
        ELSE 'otros'
    END AS tipo_operacion,

    precio,
    moneda,
    expensas,
    ambientes,
    metros_cuadrados_totales,
    metros_cuadrados_cubiertos,
    antiguedad,

    CASE 
        WHEN estado IS NULL OR estado IN ('null','0','-1','-2')
            THEN 'sin_informacion'

        WHEN estado RLIKE 'excelente|impecable|premium' 
            THEN 'excelente'

        WHEN estado RLIKE  'buen' 
            THEN 'bueno'

        WHEN estado RLIKE  'a_refaccionar|reciclado' 
            THEN 'a_refaccionar'

        WHEN estado RLIKE  'estrenar|nuevo' 
            THEN'a_estrenar'

        WHEN estado RLIKE  'regular' 
            THEN 'regular'

        WHEN estado RLIKE 'en preventa|en_construccion|terminada'
            THEN 'en_desarrollo'

        ELSE 'sin_especificar'
    END AS estado,

     url
    
    FROM datos_limpios
),

--Deduplicación
datos_deduplicados AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (
            PARTITION BY url
            ORDER BY precio
        ) AS rn
    FROM estandarizacion_columnas
),

--Calcular métricas derivadas
datos_con_metricas AS (
    SELECT 
        zona,
        ciudad,
        tipo_operacion,
        precio,
        moneda,
        expensas,
        ambientes,
        metros_cuadrados_totales,
        metros_cuadrados_cubiertos,
        antiguedad,
        estado,
        url,
        -- Métrica calculada: precio por m2
        ROUND(precio / metros_cuadrados_totales, 2) as precio_por_m2
    FROM datos_deduplicados
    WHERE rn = 1
)


--Insert final a Silver
SELECT 
    zona,
    ciudad,
    tipo_operacion,
    precio,
    moneda,
    expensas,
    ambientes,
    metros_cuadrados_totales,
    metros_cuadrados_cubiertos,
    antiguedad,
    estado,
    url,
    precio_por_m2,
    'bronze_EDA' AS _source_table,
    CURRENT_TIMESTAMP() AS _processing_timestamp
FROM datos_con_metricas
) s

ON t.url = s.url

WHEN MATCHED AND COALESCE(t.precio,0) <> COALESCE(s.precio,0) THEN
UPDATE SET
    t.precio = s.precio,
    t.moneda = s.moneda,
    t.expensas = s.expensas,
    t.precio_por_m2 = s.precio_por_m2

WHEN NOT MATCHED THEN
INSERT (
    zona,
    ciudad,
    tipo_operacion,
    precio,
    moneda,
    expensas,
    ambientes,
    metros_cuadrados_totales,
    metros_cuadrados_cubiertos,
    antiguedad,
    estado,
    url,
    precio_por_m2,
    _source_table,
    _processing_timestamp
)
VALUES (
    s.zona,
    s.ciudad,
    s.tipo_operacion,
    s.precio,
    s.moneda,
    s.expensas,
    s.ambientes,
    s.metros_cuadrados_totales,
    s.metros_cuadrados_cubiertos,
    s.antiguedad,
    s.estado,
    s.url,
    s.precio_por_m2,
    s._source_table,
    CURRENT_TIMESTAMP()
);